# Explainable Fake News Detection with DistilBERT and Uncertainty

**Machine Learning subject project**

A fine-tuned **DistilBERT** model that classifies a news article as **FAKE** or **REAL**, and also:

1. **Says when it is unsure.** Monte Carlo Dropout gives a confidence score, and low-confidence articles are labelled **UNCERTAIN** instead of being forced into FAKE or REAL.
2. **Explains its answer.** LIME highlights the words that pushed the prediction.
3. **Is tested honestly.** It is trained on political news and then tested on COVID-19 health posts it has never seen.

| # | Section |
|---|---------|
| 1 | Setup |
| 2 | Load the data |
| 3 | Clean the data (remove "giveaway" patterns) |
| 4 | Baseline: TF-IDF + Logistic Regression |
| 5 | Fine-tune DistilBERT |
| 6 | Evaluate on the test set |
| 7 | Uncertainty with MC Dropout |
| 8 | Explain predictions with LIME |
| 9 | Domain shift: test on COVID-19 news |
| 10 | Summary and demo |

**Run time:** about 15–20 minutes on a laptop GPU (RTX 3050) or a Colab T4. Every table and figure is saved in `results/`.

## 1. Setup

In [ ]:
try:
    import lime
except ImportError:          # Google Colab does not have LIME pre-installed
    %pip install -q lime

In [ ]:
import os, re, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from lime.lime_text import LimeTextExplainer
import warnings; warnings.filterwarnings("ignore")

CFG = dict(
    model_name     = "distilbert-base-uncased",
    max_len        = 256,     # tokens read per article
    batch_size     = 16,      # use 8 if you get "CUDA out of memory"
    epochs         = 2,
    lr             = 2e-5,
    train_size     = 10000,   # training articles (None = all ~27,000, about 3x slower)
    val_size       = 2000,
    test_size      = 3000,
    mc_passes      = 10,      # forward passes for MC Dropout
    uncertain_rate = 0.05,    # flag the least confident 5% as UNCERTAIN
    seed           = 42,
)

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"       # mixed precision = faster training on GPU
print("Device:", torch.cuda.get_device_name(0) if USE_AMP else "CPU (no GPU found, training will be slow)")

os.makedirs("results/figures", exist_ok=True)
LABELS = {0: "REAL", 1: "FAKE"}

## 2. Load the data

* **Training data:** Kaggle *Fake and Real News* dataset (ISOT). `True.csv` holds Reuters articles and `Fake.csv` holds articles from unreliable sites. Most are about 2016–17 US politics.
* **Cross-domain test:** COVID-19 Fake News dataset (Constraint@AAAI 2021), test split: 2,140 short social-media posts from 2020.

The files are downloaded automatically. If a download fails, put `True.csv`, `Fake.csv` and `english_test_with_labels.csv` next to this notebook.

**Labels: 0 = REAL, 1 = FAKE.**

In [ ]:
ISOT_URL  = "https://raw.githubusercontent.com/laxmimerit/fake-real-news-dataset/main/data/"
COVID_URL = "https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/english_test_with_labels.csv"

def read_csv(name, url):
    return pd.read_csv(name) if os.path.exists(name) else pd.read_csv(url)

true_df = read_csv("True.csv", ISOT_URL + "True.csv"); true_df["label"] = 0
fake_df = read_csv("Fake.csv", ISOT_URL + "Fake.csv"); fake_df["label"] = 1
news = pd.concat([true_df, fake_df], ignore_index=True)
news["text"] = news["text"].fillna("").astype(str)

covid = read_csv("english_test_with_labels.csv", COVID_URL).rename(columns={"tweet": "text"})
covid["label"] = (covid["label"].str.lower() == "fake").astype(int)

print("ISOT news :", len(news), news.label.map(LABELS).value_counts().to_dict())
print("COVID-19  :", len(covid), covid.label.map(LABELS).value_counts().to_dict())
news[["title", "text", "label"]].head()

## 3. Clean the data

This dataset has a known problem: the text contains **giveaway patterns** that reveal *where* an article came from, not whether it is true. For example, almost every REAL article starts with "WASHINGTON (Reuters) -". A model can score about 99% just by spotting these patterns, without learning anything about fake news.

We check for these patterns, remove them, and drop very short and duplicate articles. We use only the article `text`, not the `title` or `subject` columns, which have the same problem.

In [ ]:
GIVEAWAYS = {
    "(Reuters)":            r"\(Reuters\)",
    "Featured image / via": r"(?i)featured image|image via",
    "Getty Images":         r"(?i)getty images",
    "web link":             r"http|www\.|pic\.twitter",
    "Twitter @handle":      r"@\w+",
    "curly apostrophe ’":   r"’",
}

def giveaway_table(texts, labels):
    return pd.DataFrame({name: {"% of REAL": 100 * texts[labels == 0].str.contains(p).mean(),
                                "% of FAKE": 100 * texts[labels == 1].str.contains(p).mean()}
                         for name, p in GIVEAWAYS.items()}).T.round(1)

giveaway_table(news.text, news.label)

In [ ]:
CLEAN_RULES = [
    (r"^.{0,150}?\(Reuters\)\s*[-–—]\s*", ""),                       # "WASHINGTON (Reuters) - "
    (r"(?i)featured\s+(image|photo).*$", ""),                         # image credit at the end of fake articles
    (r"(?i)\b(photo|image|video)s?\s*(credit)?\s*:?\s*(via|by|courtesy)\b[^.]{0,120}", " "),
    (r"(?i)getty\s*images|\breuters\b", " "),                        # leftover source names
    (r"https?://\S+|www\.\S+|pic\.twitter\.com/\S+|\S+\.com/\S+", " "),   # links
    (r"@\w+", " "),                                                   # Twitter handles
    (r"[’‘'`´]", " "),                                                # write apostrophes the same way in both classes
    (r"[“”]", '"'),
    (r"\s+", " "),
]
CLEAN_RULES = [(re.compile(p), r) for p, r in CLEAN_RULES]

def clean_text(s):
    for pattern, replacement in CLEAN_RULES:
        s = pattern.sub(replacement, str(s))
    return s.strip()

news["clean"] = news["text"].map(clean_text)
data = news[news["clean"].str.split().str.len() >= 20]       # drop empty / very short articles
data = data.drop_duplicates("clean").reset_index(drop=True)   # drop duplicates
print(f"{len(news)} articles -> {len(data)} after cleaning")

after = giveaway_table(data.clean, data.label)
before_after = giveaway_table(news.text, news.label).join(after, rsuffix=" (after)")
before_after.to_csv("results/giveaways_before_after.csv")
before_after

After cleaning, all the giveaway patterns are gone from both classes.

### Train / validation / test split

The data is split 70 / 15 / 15 with the same FAKE/REAL balance in each part. To keep the run short on a laptop, we then use a random sample of each part (sizes set in `CFG`).
The **validation** set is used only to choose the UNCERTAIN threshold; the **test** set is used only for the final results.

In [ ]:
train_df, rest = train_test_split(data, test_size=0.30, stratify=data.label, random_state=CFG["seed"])
val_df, test_df = train_test_split(rest, test_size=0.50, stratify=rest.label, random_state=CFG["seed"])

def take(df, n):
    return df if n is None or n >= len(df) else df.sample(n, random_state=CFG["seed"])
train_df, val_df, test_df = take(train_df, CFG["train_size"]), take(val_df, CFG["val_size"]), take(test_df, CFG["test_size"])

covid["clean"] = covid["text"].map(clean_text)
covid = covid[covid["clean"].str.split().str.len() >= 3].reset_index(drop=True)

y_val, y_test, y_covid = val_df.label.values, test_df.label.values, covid.label.values
for name, d in [("train", train_df), ("validation", val_df), ("test", test_df), ("COVID-19", covid)]:
    print(f"{name:10s} {len(d):6d} articles   fake share = {d.label.mean():.2f}")

## 4. Baseline: TF-IDF + Logistic Regression

A simple classical model, so we can see how much the transformer adds.

**Metrics** (FAKE is the positive class):
* **Accuracy:** share of correct predictions.
* **Precision:** of the articles called FAKE, how many really are fake.
* **Recall:** of the fake articles, how many were caught.
* **F1:** balance of precision and recall.
* **ROC-AUC:** how well the scores rank fake above real (1.0 = perfect).
* **ECE (Expected Calibration Error):** does a confidence of 90% really mean 90% correct? Predictions are grouped by confidence, and ECE is the average gap between confidence and accuracy. 0 = perfectly trustworthy confidence.

In [ ]:
def ece(y, p_fake, n_bins=10):
    """Expected Calibration Error: average |accuracy - confidence| over confidence bins."""
    conf = np.maximum(p_fake, 1 - p_fake)
    correct = ((p_fake >= 0.5) == y)
    bins = np.minimum(((conf - 0.5) / 0.5 * n_bins).astype(int), n_bins - 1)
    return sum((bins == b).mean() * abs(correct[bins == b].mean() - conf[bins == b].mean())
               for b in range(n_bins) if (bins == b).any())

def evaluate(y, p_fake, name):
    pred = (p_fake >= 0.5).astype(int)
    return {"model": name,
            "accuracy":  accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall":    recall_score(y, pred, zero_division=0),
            "f1":        f1_score(y, pred, zero_division=0),
            "roc_auc":   roc_auc_score(y, p_fake),
            "ECE":       ece(y, p_fake)}

baseline = make_pipeline(TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, C=4))
baseline.fit(train_df.clean, train_df.label)
p_base = baseline.predict_proba(test_df.clean)[:, 1]

results = [evaluate(y_test, p_base, "TF-IDF + LR")]
pd.DataFrame(results).round(4)

## 5. Fine-tune DistilBERT

**DistilBERT** is a smaller, faster version of BERT (6 layers, 66 million parameters) that was pre-trained on English Wikipedia and books. We add a FAKE/REAL classification layer on top and fine-tune the whole model on our articles. This is **transfer learning**: the model already understands English, so it only has to learn the new task.

Settings: first 256 tokens of each article, batch size 16, learning rate 2e-5 with warm-up, AdamW optimiser, 2 epochs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=2).to(DEVICE)

def tokenize(texts):
    return tokenizer(list(texts), truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt")

@torch.no_grad()
def predict_proba(texts, mc_dropout=False, batch_size=64):
    """Return P(fake) for each text.
    mc_dropout=True keeps dropout switched ON, so every call gives a slightly different answer."""
    model.train(mc_dropout)          # train mode only switches dropout on (DistilBERT has no batch-norm)
    texts, out = list(texts), []
    for i in range(0, len(texts), batch_size):
        enc = tokenize(texts[i:i + batch_size]).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**enc).logits.float()
        out.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    model.eval()
    return np.concatenate(out)

In [ ]:
loader = DataLoader(list(zip(train_df.clean, train_df.label)), batch_size=CFG["batch_size"], shuffle=True,
                    collate_fn=lambda batch: (tokenize([t for t, _ in batch]), torch.tensor([l for _, l in batch])))
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
total_steps = len(loader) * CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history = []
for epoch in range(1, CFG["epochs"] + 1):
    model.train(); start, total_loss = time.time(), 0.0
    for step, (enc, labels) in enumerate(loader, 1):
        enc, labels = enc.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss = model(**enc, labels=labels).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total_loss += loss.item()
        if step % 100 == 0:
            print(f"  epoch {epoch}  step {step}/{len(loader)}  loss {total_loss / step:.4f}")
    val_acc = accuracy_score(y_val, predict_proba(val_df.clean) >= 0.5)
    history.append({"epoch": epoch, "train loss": total_loss / len(loader), "validation accuracy": val_acc})
    print(f"epoch {epoch} done in {time.time() - start:.0f}s | train loss {total_loss / len(loader):.4f} | val accuracy {val_acc:.4f}")

pd.DataFrame(history).round(4)

## 6. Evaluate on the test set

In [ ]:
p_bert = predict_proba(test_df.clean)
results.append(evaluate(y_test, p_bert, "DistilBERT"))
pd.DataFrame(results).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, p) in zip(axes, [("TF-IDF + LR", p_base), ("DistilBERT", p_bert)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p >= 0.5), display_labels=["REAL", "FAKE"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.suptitle("Confusion matrices: ISOT test set")
plt.tight_layout(); plt.savefig("results/figures/confusion_matrices.png", dpi=150); plt.show()

## 7. Uncertainty with Monte Carlo Dropout

Dropout normally randomly switches off some neurons **during training only**. **Monte Carlo (MC) Dropout** (Gal & Ghahramani, 2016) keeps dropout **on at prediction time** and runs each article through the model several times (here 10). Each pass uses a slightly different network:

* if the passes **agree**, the model is confident;
* if they **disagree**, the model is unsure.

We use the **average** of the passes as the prediction and **confidence = max(P(fake), P(real))**.

**UNCERTAIN flag:** on the *validation* set we pick a threshold τ so that the least confident 5% are flagged. Then we apply the same τ to the test set. A flagged article would go to a human fact-checker instead of getting an automatic label.

In [ ]:
def mc_dropout(texts, passes=CFG["mc_passes"]):
    runs = np.stack([predict_proba(texts, mc_dropout=True) for _ in range(passes)])   # (passes, n_articles)
    p_mean = runs.mean(axis=0)
    confidence = np.maximum(p_mean, 1 - p_mean)
    return p_mean, runs.std(axis=0), confidence

start = time.time()
p_val_mc, _, conf_val = mc_dropout(val_df.clean)
p_mc, std_mc, conf_mc = mc_dropout(test_df.clean)
print(f"MC Dropout done in {time.time() - start:.0f}s")

tau = np.quantile(conf_val, CFG["uncertain_rate"])
print(f"UNCERTAIN threshold tau = {tau:.4f}  (confidence below this -> UNCERTAIN)")

results.append(evaluate(y_test, p_mc, "DistilBERT + MC Dropout"))
pd.DataFrame(results).round(4)

### Does low confidence really mean "likely wrong"?

This is the key check. If the confidence score is useful:
* accuracy on **confident** predictions should be higher than overall accuracy;
* accuracy on **UNCERTAIN** predictions should be clearly lower;
* the flag should catch a large share of all mistakes while flagging only 5% of articles.

We also report the **error-detection AUROC**: how well (1 − confidence) separates wrong predictions from right ones (0.5 = no better than chance, 1.0 = perfect).

In [ ]:
def uncertainty_report(y, p_fake, confidence, name):
    wrong = (p_fake >= 0.5) != y
    flagged = confidence < tau
    return {"data": name,
            "% flagged UNCERTAIN":        100 * flagged.mean(),
            "accuracy (all)":             1 - wrong.mean(),
            "accuracy (confident only)":  1 - wrong[~flagged].mean() if (~flagged).any() else np.nan,
            "accuracy (UNCERTAIN only)":  1 - wrong[flagged].mean() if flagged.any() else np.nan,
            "% of mistakes flagged":      100 * (wrong & flagged).sum() / max(wrong.sum(), 1),
            "error-detection AUROC":      roc_auc_score(wrong, 1 - confidence) if 0 < wrong.sum() < len(y) else np.nan}

uncertainty = [uncertainty_report(y_test, p_mc, conf_mc, "ISOT test")]
pd.DataFrame(uncertainty).round(4)

In [ ]:
def reliability_diagram(ax, y, p_fake, title, n_bins=10):
    conf = np.maximum(p_fake, 1 - p_fake)
    correct = ((p_fake >= 0.5) == y)
    bins = np.minimum(((conf - 0.5) / 0.5 * n_bins).astype(int), n_bins - 1)
    used = [b for b in range(n_bins) if (bins == b).any()]
    ax.plot([0.5, 1], [0.5, 1], "--", color="gray", label="perfectly calibrated")
    ax.plot([conf[bins == b].mean() for b in used], [correct[bins == b].mean() for b in used], "o-", label="model")
    ax.set(xlabel="confidence", ylabel="accuracy", xlim=(0.5, 1.01), ylim=(0, 1.02),
           title=f"{title}\nECE = {ece(y, p_fake):.4f}")
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
reliability_diagram(axes[0], y_test, p_bert, "DistilBERT (normal softmax)")
reliability_diagram(axes[1], y_test, p_mc, "DistilBERT + MC Dropout")
right = (p_mc >= 0.5) == y_test
axes[2].hist(conf_mc[right], bins=30, alpha=0.6, label="correct", density=True)
axes[2].hist(conf_mc[~right], bins=30, alpha=0.6, label="wrong", density=True)
axes[2].axvline(tau, color="k", ls="--", label="UNCERTAIN threshold")
axes[2].set(xlabel="MC Dropout confidence", ylabel="density", title="Confidence of correct vs wrong predictions")
axes[2].legend()
plt.tight_layout(); plt.savefig("results/figures/confidence_vs_accuracy.png", dpi=150); plt.show()

## 8. Explain predictions with LIME

**LIME** (Local Interpretable Model-agnostic Explanations, Ribeiro et al. 2016) explains one prediction at a time:
1. it makes a few hundred copies of the article with random words removed;
2. it asks DistilBERT for a prediction on each copy;
3. it fits a small linear model to see which words changed the prediction most.

Words with **positive** weight push towards **FAKE**, and **negative** weight push towards **REAL**. We explain one confident FAKE prediction, one confident REAL prediction and the least confident article.

In [ ]:
explainer = LimeTextExplainer(class_names=["REAL", "FAKE"], random_state=CFG["seed"])

def lime_predict(texts):                   # LIME needs a (n, 2) probability matrix
    p = predict_proba(texts)
    return np.c_[1 - p, p]

def first_words(text, n=150):              # DistilBERT reads only the first 256 tokens anyway
    return " ".join(str(text).split()[:n])

pred_mc = (p_mc >= 0.5).astype(int)
examples = {"confident FAKE":  np.where(pred_mc == 1)[0][np.argmax(conf_mc[pred_mc == 1])],
            "confident REAL":  np.where(pred_mc == 0)[0][np.argmax(conf_mc[pred_mc == 0])],
            "most UNCERTAIN":  np.argmin(conf_mc)}

for tag, i in examples.items():
    text = first_words(test_df.clean.iloc[i])
    exp = explainer.explain_instance(text, lime_predict, num_features=10, num_samples=500)
    print(f"--- {tag} | true = {LABELS[y_test[i]]} | predicted = {LABELS[pred_mc[i]]} | "
          f"P(fake) = {p_mc[i]:.3f} | confidence = {conf_mc[i]:.3f}")
    print(text[:400], "...")
    exp.as_pyplot_figure()
    plt.title(f"LIME: {tag} (true = {LABELS[y_test[i]]})")
    plt.tight_layout(); plt.savefig(f"results/figures/lime_{tag.replace(' ', '_').lower()}.png", dpi=150); plt.show()

**How to read this:** look at whether the top words are about the *content* (claims, people, events) or about *writing style* (words like "said", "Tuesday", "via", emotional words). Because every REAL article in this dataset comes from Reuters, the model partly learns news-agency style. LIME makes this visible, which is exactly why explainability matters.

## 9. Domain shift: from political news to COVID-19

The model was trained on long political news articles from 2016–17. The COVID-19 posts are different in **topic** (health), **time** (2020) and **format** (short social-media posts).

* If the model really understood misinformation, accuracy would stay high.
* If it mostly learned the *style* of its training data, accuracy will drop a lot.
* Ideally the UNCERTAIN flag should fire **more often** on this unfamiliar data.

In [ ]:
p_base_cov = baseline.predict_proba(covid.clean)[:, 1]
p_bert_cov = predict_proba(covid.clean)
p_mc_cov, _, conf_mc_cov = mc_dropout(covid.clean)

covid_results = [evaluate(y_covid, p_base_cov, "TF-IDF + LR"),
                 evaluate(y_covid, p_bert_cov, "DistilBERT"),
                 evaluate(y_covid, p_mc_cov,   "DistilBERT + MC Dropout")]

in_dom, cross = pd.DataFrame(results).set_index("model"), pd.DataFrame(covid_results).set_index("model")
domain_shift = pd.DataFrame({"ISOT accuracy": in_dom.accuracy, "COVID-19 accuracy": cross.accuracy,
                             "accuracy drop (points)": 100 * (in_dom.accuracy - cross.accuracy),
                             "ISOT F1": in_dom.f1, "COVID-19 F1": cross.f1,
                             "ISOT ECE": in_dom.ECE, "COVID-19 ECE": cross.ECE})
domain_shift.round(4)

In [ ]:
uncertainty.append(uncertainty_report(y_covid, p_mc_cov, conf_mc_cov, "COVID-19"))
uncertainty_table = pd.DataFrame(uncertainty)
display(uncertainty_table.round(4))
print(f"Share of COVID-19 posts predicted FAKE: {(p_mc_cov >= 0.5).mean():.1%}  (true share: {y_covid.mean():.1%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
domain_shift[["ISOT accuracy", "COVID-19 accuracy"]].plot.bar(ax=axes[0], rot=0, ylim=(0, 1.05),
                                                             title="Accuracy: in-domain vs COVID-19")
axes[0].axhline(0.5, color="gray", ls="--", lw=1)
reliability_diagram(axes[1], y_covid, p_mc_cov, "MC Dropout on COVID-19")
plt.tight_layout(); plt.savefig("results/figures/domain_shift.png", dpi=150); plt.show()

## 10. Summary and demo

In [ ]:
summary = pd.concat([pd.DataFrame(results).assign(test_set="ISOT"),
                     pd.DataFrame(covid_results).assign(test_set="COVID-19")])
summary.round(4).to_csv("results/metrics.csv", index=False)
domain_shift.round(4).to_csv("results/domain_shift.csv")
uncertainty_table.round(4).to_csv("results/uncertainty.csv", index=False)
print("Saved:", sorted(os.listdir("results")), "\nFigures:", sorted(os.listdir("results/figures")))
summary.round(4)

In [ ]:
def check_news(text):
    """Classify any piece of text: FAKE / REAL / UNCERTAIN, with confidence and key words."""
    text = first_words(clean_text(text))
    p_fake, std, conf = mc_dropout([text])
    verdict = "UNCERTAIN (send to a human fact-checker)" if conf[0] < tau else ("FAKE" if p_fake[0] >= 0.5 else "REAL")
    exp = explainer.explain_instance(text, lime_predict, num_features=6, num_samples=300)
    print(f"Verdict    : {verdict}")
    print(f"P(fake)    : {p_fake[0]:.3f}  (± {std[0]:.3f} across {CFG['mc_passes']} MC Dropout passes)")
    print(f"Confidence : {conf[0]:.3f}  (UNCERTAIN below {tau:.3f})")
    print("Key words  : " + ", ".join(f"{w} -> {'FAKE' if v > 0 else 'REAL'}" for w, v in exp.as_list()))

check_news("The Senate voted 52-48 on Tuesday to approve the budget resolution, sending the measure to the House, "
           "where Republican leaders said they expected a vote later this week.")
print("-" * 80)
check_news("BREAKING: Doctors are FURIOUS after a mom discovered this one weird trick that cures diabetes overnight! "
           "The mainstream media won t tell you the truth. Share before it gets deleted!")

## Conclusions

Fill these in from your results:

1. **Accuracy:** both models score very high on the ISOT test set, even after the giveaway patterns are removed. DistilBERT is compared with the TF-IDF baseline in the table above.
2. **Uncertainty:** look at the uncertainty table. Is accuracy on UNCERTAIN articles much lower than on confident ones? What share of all mistakes does the flag catch while flagging only about 5% of articles?
3. **Calibration:** compare ECE for normal softmax and MC Dropout. Lower ECE means the confidence scores are more trustworthy.
4. **Explainability:** LIME shows which words drive each decision, including style words that come from the news source rather than the facts.
5. **Domain shift:** accuracy drops sharply on COVID-19 posts. The model learned the style of 2016–17 political news, not a general sense of "truth". This is an honest and important limitation.

**Limitations:** the model checks *writing style*, not facts; every REAL training article comes from one source (Reuters); only the first 256 tokens are read; MC Dropout can still be over-confident on very different data.

**References:** Sanh et al. 2019 (DistilBERT) · Gal & Ghahramani 2016 (MC Dropout) · Ribeiro et al. 2016 (LIME) · Guo et al. 2017 (calibration / ECE) · Patwa et al. 2021 (COVID-19 dataset) · Ahmed et al. 2017 (ISOT dataset)